In [13]:
import pandas as pd
import os

# Set your local folder path
folder_path = r"C:\Users\lervi\Downloads\Vanguard_Merge_Files"

print("Loading files from:", folder_path)

# 1. Load the files
df_exp = pd.read_csv(os.path.join(folder_path, 'df_final_experiment_clients.txt'))
df_demo = pd.read_csv(os.path.join(folder_path, 'df_final_demo (1).txt'))
df_web = pd.read_csv(os.path.join(folder_path, 'df_web_clean.csv'))

# 2. Merge experiment and demographics on 'client_id'
master = pd.merge(df_exp, df_demo, on='client_id', how='outer')

# 3. Aggregate web interaction metrics per client
web_summary = df_web.groupby('client_id')['process_step'].agg(
    total_interactions='count',
    has_confirmed=lambda x: (x == 'confirm').any()
).reset_index()

# 4. Merge web summary into the master dataset
df_vanguard_master = pd.merge(master, web_summary, on='client_id', how='left')

# 5. Save the final file into your current notebook directory
df_vanguard_master.to_csv('df_vanguard_master_dataset.csv', index=False)

print("SUCCESS! 'df_vanguard_master_dataset.csv' has been created.")
print("Total rows and columns:", df_vanguard_master.shape)

Loading files from: C:\Users\lervi\Downloads\Vanguard_Merge_Files
SUCCESS! 'df_vanguard_master_dataset.csv' has been created.
Total rows and columns: (70609, 12)


In [14]:
import pandas as pd

# Load your dataset
df = pd.read_csv('df_vanguard_master_dataset.csv')

# Print the columns as a list
print(df.columns.tolist())

['client_id', 'Variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'total_interactions', 'has_confirmed']


### 📋 Column Breakdown & What They Mean

| Icon | Column Name | Source File | What it Represents (Simple Terms) |
| :---: | :--- | :--- | :--- |
| 🆔 | **`client_id`** | `df_final_experiment_clients.txt`, `df_final_demo (1).txt`, & `df_web_clean.csv` | The customer's unique ID number used to match everything together across all files. |
| 🧪 | **`Variation`** | `df_final_experiment_clients.txt` | Tells us which website version they tested: **Test** (new design) or **Control** (old design). |
| 📅 | **`clnt_tenure_yr`** | `df_final_demo (1).txt` | How many **years** the customer has been with Vanguard. |
| ⏱️ | **`clnt_tenure_mnth`** | `df_final_demo (1).txt` | How many total **months** the customer has been with Vanguard. |
| 👤 | **`clnt_age`** | `df_final_demo (1).txt` | The **age** of the customer. |
| ⚧️ | **`gendr`** | `df_final_demo (1).txt` | The **gender** of the customer (e.g., Male, Female, or Unknown). |
| 💳 | **`num_accts`** | `df_final_demo (1).txt` | How many **open accounts** the customer has. |
| 💰 | **`bal`** | `df_final_demo (1).txt` | The **total balance** (money) across the customer's accounts. |
| 📞 | **`calls_6_mnth`** | `df_final_demo (1).txt` | How many times they **called customer support** in the last 6 months. |
| 💻 | **`logons_6_mnth`** | `df_final_demo (1).txt` | How many times they **logged into their online account** in the last 6 months. |
| 🖱️ | **`total_interactions`** | `df_web_clean.csv` (Calculated) | How many **total clicks or actions** the customer made on the website. |
| ✅ | **`has_confirmed`** | `df_web_clean.csv` (Calculated) | A simple **Yes or No** (True/False): *Did this customer successfully finish the whole website process?* |

---

### ⚙️ What the Code Is Doing (Step-by-Step)

- 📂 **Step 1 (Loading files)**: The script opens `df_final_experiment_clients.txt`, `df_final_demo (1).txt`, and `df_web_clean.csv` from your local folder.
- 🧩 **Step 2 (First Merge)**: It glues the experiment assignments and customer profile details side-by-side using their ID badge (`client_id`).
- 📊 **Step 3 (Website Summary)**: It looks through all individual website clicks in `df_web_clean.csv`, counts how many total actions each person took, and checks if they reached the final confirmation screen.
- 💾 **Step 4 (Final Merge & Save)**: It attaches those website results to your main customer table and saves everything into one clean master spreadsheet (`df_vanguard_master_dataset.csv`).

In [16]:
import pandas as pd

# Load your dataset
df = pd.read_csv('df_vanguard_master_dataset.csv')

# Display the first 5 rows as a formatted table
display(df.head())

,client_id,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,total_interactions,has_confirmed
0,169,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0,5,True
1,555,Test,3.0,46.0,29.5,U,2.0,25454.66,2.0,6.0,5,True
2,647,Test,12.0,151.0,57.5,M,2.0,30525.80,0.0,4.0,5,True
3,722,NaN,11.0,143.0,59.5,F,2.0,22466.17,1.0,1.0,9,True
4,934,Test,9.0,109.0,51.0,F,2.0,32522.88,0.0,3.0,4,False


### Why Weren't the Process Steps Added as Individual Columns Originally?

Think of your data as two different types of records:

1. **The Master Table (Your Client Profiles):**
   Imagine a spreadsheet where each row represents one unique person (one `client_id`). This person has static details that don't change based on website clicks, like their age (`clnt_age`), account balance (`bal`), tenure (`clnt_tenure_yr`), and whether they are in the Test or Control group (`Variation`).   

2. **The Web Log Table (`df_web_clean.csv`):**
   Imagine a detailed activity log or a receipt history where each row represents a single click or action on a website. Because a single client can visit the website multiple times—and might click back and forth between steps (for example, moving forward to `step_3`, going back to `step_2`, or starting over completely)—one client will have many rows of web clicks.

### The Problem with "Gluing" Them Together Directly:
If we took every single website click and forced it into a single row per client, your client's profile data (like their age and balance) would have to be duplicated hundreds of times for every single click they made. This would make your master dataset bloated, messy, and extremely difficult to analyze.

In [20]:
import pandas as pd
import os

# Your folder path
folder_path = r"C:\Users\lervi\Downloads\Vanguard_Merge_Files"

# Load the web data using the full path
df_web = pd.read_csv(os.path.join(folder_path, 'df_web_clean.csv'))

# Count how many times each step was visited overall
print(df_web['process_step'].value_counts())

process_step
start      234999
step_1     162797
step_2     132750
step_3     111589
confirm    102506
Name: count, dtype: int64


In [21]:
import pandas as pd
import os

folder_path = r"C:\Users\lervi\Downloads\Vanguard_Merge_Files"

# 1. Load the current master dataset and the web data
df_master = pd.read_csv('df_vanguard_master_dataset.csv')
df_web = pd.read_csv(os.path.join(folder_path, 'df_web_clean.csv'))

# 2. Pivot the web data to count occurrences of each step per client
df_steps = df_web.pivot_table(
    index='client_id', 
    columns='process_step', 
    values='visit_id', 
    aggfunc='count', 
    fill_value=0
).reset_index()

# Rename columns to be descriptive (e.g., 'step_1_count', 'confirm_count')
df_steps.columns = ['client_id'] + [f"{col}_count" for col in df_steps.columns if col != 'client_id']

# 3. Merge these new step columns into your master dataset
df_master_updated = pd.merge(df_master, df_steps, on='client_id', how='left')

# Fill NaN with 0 for clients who had no web activity
step_cols = [col for col in df_master_updated.columns if '_count' in col]
df_master_updated[step_cols] = df_master_updated[step_cols].fillna(0)

# 4. Save the updated master dataset
df_master_updated.to_csv('df_vanguard_master_dataset.csv', index=False)

print("Updated master dataset columns:", df_master_updated.columns.tolist())

Updated master dataset columns: ['client_id', 'Variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'total_interactions', 'has_confirmed', 'confirm_count', 'start_count', 'step_1_count', 'step_2_count', 'step_3_count']


In [23]:
import pandas as pd

# Load your master dataset
df = pd.read_csv('df_vanguard_master_dataset.csv')

# Count the number of clients in each variation group (including unassigned clients if any)
print(df['Variation'].value_counts(dropna=False))

Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64


In [24]:
import pandas as pd

# Load your master dataset
df = pd.read_csv('df_vanguard_master_dataset.csv')

# Drop missing values in the Variation column to get only those in the experiment
experiment_clients = df['Variation'].dropna()

print("Total clients in the A/B test (Test + Control):", len(experiment_clients))
print("\nBreakdown by group:")
print(df['Variation'].value_counts())

Total clients in the A/B test (Test + Control): 50500

Breakdown by group:
Variation
Test       26968
Control    23532
Name: count, dtype: int64
